# Error correction across two CPUs with Backline

In this notebook you'll run a **logical** quantum circuit, protected by the Steane code. A controller runs the circuit. A coprocessor decodes the errors. **Backline** connects the two.

Adapted from Xanadu's [How to use PennyLane and Backline](https://pennylane.ai/demos/backline) by Josh Izaac, Mehrdad Malek, Joseph Lee and Kalman Szenes (Apache 2.0).

**Time:** about a minute to the first result, even on the free 2 vCPU instance. The last run, with three logical qubits, is optional and needs at least 4 vCPUs.

## 1. Check your setup

Pick the **PennyLane + Backline** kernel (top right), then run this cell. It ends with "Setup OK" when you're ready.

In [ ]:
import os
import shutil
import sys
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

# One simulator thread is fastest for circuits this small. Set it before importing PennyLane.
os.environ.setdefault("OMP_NUM_THREADS", "1")

try:
    import catalyst
    import pennylane as qp
except ImportError as err:
    raise SystemExit(
        f"{err}\nThis kernel doesn't have the Backline packages. "
        "Pick the PennyLane + Backline kernel at the top right and run this cell again."
    )

if not hasattr(qp, "Backline"):
    raise SystemExit(
        f"PennyLane {qp.__version__} doesn't include Backline. "
        "Pick the PennyLane + Backline kernel at the top right and run this cell again."
    )

# Catalyst links the compiled program with a C compiler.
if not (shutil.which("gcc") or shutil.which("cc") or shutil.which("clang")):
    raise SystemExit("No C compiler found. Catalyst needs gcc or clang on PATH.")

# The Steane decoder ships inside the Catalyst package.
STEANE_LIB = Path(catalyst.__file__).parent / "lib" / "libsteane_coprocessor_cpu.so"
if not STEANE_LIB.exists():
    raise SystemExit(f"Steane decoder library not found at {STEANE_LIB}")

print("Python    ", sys.version.split()[0])
for package in ("pennylane", "pennylane-lightning", "pennylane-catalyst"):
    try:
        print(f"{package:20s}", version(package))
    except PackageNotFoundError:
        print(f"{package:20s}", "not installed")
print("Setup OK. You're good to go.")

## 2. The three parts of a Backline

- **Controller:** runs the circuit, measures, and sends each error syndrome off to be decoded. Here it's a CPU driving the Lightning simulator.
- **Coprocessor:** receives a syndrome and works out which qubit to fix. Here it's a second CPU role, running a pre-compiled Steane decoder.
- **Backline:** ties them together with a transport. `memcpy` passes messages through memory, because both roles live on this machine. On bigger setups the same program can use RDMA to reach a GPU or an FPGA.

You hand the Backline to a QNode where you'd normally give it a device.

In [ ]:
def make_backline(n_logical):
    """A CPU controller and a CPU decoder, joined by an in-memory transport."""
    decoder = qp.CoprocessorFunction("steane_coprocessor", lib_path=str(STEANE_LIB))
    controller = qp.Controller(
        name="cpu-controller",
        device=qp.device("lightning.qubit", wires=n_logical),
    )
    coprocessor = qp.Coprocessor(name="cpu-decoder", coprocessor_fn=decoder)
    return qp.Backline(
        controller=controller,
        coprocessors=[coprocessor],
        transport="memcpy",
        qec_code="steane",
    )

## 3. A logical Bell pair

The circuit below is written on **logical** qubits. `qec_code="steane"` tells Catalyst to encode each one in seven physical qubits, and to add rounds of error correction around the gates. You don't write any of that. Each decode inside those rounds becomes a trip to the coprocessor.

Compiling takes about 25 seconds the first time. The 10 shots take a few seconds more.

In [ ]:
bell_backline = make_backline(2)

start = time.time()


@qp.qjit(capture=True)
@qp.set_shots(10)
@qp.qnode(bell_backline, mcm_method="one-shot")
def bell():
    qp.Hadamard(0)
    qp.CNOT([0, 1])
    return qp.sample([qp.measure(0), qp.measure(1)])


print(f"compiled in {time.time() - start:.1f} s")

start = time.time()
samples = bell()
print(f"10 shots in {time.time() - start:.1f} s")
print(samples)

Each row is one shot. The two logical qubits are entangled, so every row should be `0 0` or `1 1`. Let's check, and count how often each one came up.

In [ ]:
rows = [tuple(int(bit) for bit in row) for row in samples]
agree = sum(1 for a, b in rows if a == b)

print(f"{agree} of {len(rows)} shots agree")
for outcome in sorted(set(rows)):
    print(outcome, rows.count(outcome))

## 4. Optional: the three-qubit GHZ circuit

This is the circuit from Xanadu's demo. Three logical qubits are 21 data qubits plus the extra qubits that read out the errors, so simulating it is much heavier. Use an instance with at least 4 vCPUs: it takes about 50 seconds **per shot** there, and about 1 GB of memory. It runs two shots by default. Raise `SHOTS` if you have time.

In [ ]:
SHOTS = 2

ghz_backline = make_backline(3)

start = time.time()


@qp.qjit(capture=True)
@qp.set_shots(SHOTS)
@qp.qnode(ghz_backline, mcm_method="one-shot")
def ghz():
    qp.Hadamard(0)
    qp.CNOT([0, 1])
    qp.CNOT([1, 2])
    return qp.sample([qp.measure(0), qp.measure(1), qp.measure(2)])


print(f"compiled in {time.time() - start:.1f} s")

start = time.time()
ghz_samples = ghz()
print(f"{SHOTS} shots in {time.time() - start:.1f} s")
print(ghz_samples)

## Where to go next

- **Swap the decoder.** `CoprocessorFunction` points at a library, so you can give Backline one you built yourself.
- **Move to real accelerators.** [Xanadu's full demo](https://pennylane.ai/demos/backline) takes the same ideas to an AMD GPU and an FPGA.
- **Decode with AI.** [Hierarchical QEC decoding](https://pennylane.ai/demos/backline-ising-decoders) pairs NVIDIA Ising with PyMatching.
- **Read the [Backline whitepaper](https://xanadu.ai/docs/backline-whitepaper.pdf).**